<a href="https://colab.research.google.com/github/Vaishnavi943/ML_Project/blob/main/Satellite_Image_Classifier_and_Detect_Deforestation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **SATELLITE IMAGE CLASSIFIER**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os

In [ ]:
DATASET_PATH = '/content/drive/MyDrive/Dataset/EuroSAT_RGB'

In [ ]:
for item in os.listdir(DATASET_PATH):
    count = len(os.listdir(f"{DATASET_PATH}/{item}"))
    print(f"{item}  →  {count} images")

In [ ]:
BATCH_SIZE   = 32
NUM_CLASSES  = 10
NUM_EPOCHS   = 10
FINETUNE_EPOCHS = 5
LR           = 0.001
LR_FINETUNE  = 0.0001
DEVICE       = 'cuda' if torch.cuda.is_available() else 'cpu'
MODEL_SAVE   = '/content/drive/MyDrive/eurosat_resnet50.pth'

print(f"Using device: {DEVICE}")

In [ ]:
CLASS_NAMES = [
    'AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway',
    'Industrial', 'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake'
]

Transformer

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),  # extra augmentation
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

In [ ]:
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

In [ ]:
# Load Dataset & Split

full_dataset = datasets.ImageFolder(root=DATASET_PATH, transform=train_transform)
print(f"Total images: {len(full_dataset)}")
print(f"Classes found: {full_dataset.classes}")


In [ ]:
train_size = int(0.8 * len(full_dataset))
val_size   = int(0.1 * len(full_dataset))
test_size  = len(full_dataset) - train_size - val_size

In [ ]:
train_set, val_set, test_set = random_split(
    full_dataset, [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(42)
)

In [ ]:
# Apply different transforms to val/test (no augmentation)
val_set.dataset.transform  = test_transform
test_set.dataset.transform = test_transform

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_set,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_set,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train: {len(train_set)} | Val: {len(val_set)} | Test: {len(test_set)}")

In [ ]:
# ---- STEP 5: Build Model ----
model = models.resnet50(pretrained=True)

In [ ]:
# Freeze all layers
for param in model.parameters():
    param.requires_grad = False

In [ ]:
# Replace the final FC layer for 10 classes
num_features = model.fc.in_features
model.fc = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(num_features, NUM_CLASSES)
)

In [ ]:
model = model.to(DEVICE)
print("Model loaded. Final layer unfrozen, all others frozen.")

In [ ]:
# ---- STEP 6: Training Function ----
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    return running_loss / len(loader), correct / total

In [ ]:
def evaluate(model, loader, criterion):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return running_loss / len(loader), correct / total

In [ ]:
# ---- STEP 7: Phase 1 — Train Only the Final Layer ----
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=LR)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.5)



In [ ]:
best_val_acc = 0.0
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

In [ ]:
print("\n=== Phase 1: Training Final Layer ===")
for epoch in range(NUM_EPOCHS):
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion)
    val_loss, val_acc     = evaluate(model, val_loader, criterion)
    scheduler.step()

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), MODEL_SAVE)
        print(f"  ✓ Saved best model (val_acc={val_acc:.4f})")

    print(f"Epoch [{epoch+1}/{NUM_EPOCHS}] "
          f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")


=== Phase 1: Training Final Layer ===


In [ ]:
# ---- STEP 8: Phase 2 — Fine-Tune Entire Model ----
print("\n=== Phase 2: Fine-Tuning All Layers ===")
for param in model.parameters():
    param.requires_grad = True

optimizer = optim.Adam(model.parameters(), lr=LR_FINETUNE)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=FINETUNE_EPOCHS)

for epoch in range(FINETUNE_EPOCHS):
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion)
    val_loss, val_acc     = evaluate(model, val_loader, criterion)
    scheduler.step()

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), MODEL_SAVE)
        print(f"  ✓ Saved best model (val_acc={val_acc:.4f})")

    print(f"FT Epoch [{epoch+1}/{FINETUNE_EPOCHS}] "
          f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

In [ ]:
# ---- STEP 9: Final Evaluation on Test Set ----
print("\n=== Final Test Evaluation ===")
model.load_state_dict(torch.load(MODEL_SAVE))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(DEVICE)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())

print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES))

In [ ]:
# ---- STEP 10: Confusion Matrix ----
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel('Predicted', fontsize=12)
plt.ylabel('Actual', fontsize=12)
plt.title('Confusion Matrix — EuroSAT ResNet50', fontsize=14)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/confusion_matrix.png', dpi=150)
plt.show()
print("Confusion matrix saved.")

In [ ]:
# ---- STEP 11: Training Curves ----
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(history['train_loss'], label='Train Loss')
ax1.plot(history['val_loss'],   label='Val Loss')
ax1.set_title('Loss Curve'); ax1.legend(); ax1.set_xlabel('Epoch')

ax2.plot(history['train_acc'], label='Train Acc')
ax2.plot(history['val_acc'],   label='Val Acc')
ax2.set_title('Accuracy Curve'); ax2.legend(); ax2.set_xlabel('Epoch')
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/training_curves.png', dpi=150)
plt.show()
print("Training curves saved.")

In [ ]:

# Save model + class names together in one dict
torch.save({
    'model_state_dict': model.state_dict(),
    'class_names': CLASS_NAMES,
    'num_classes': NUM_CLASSES,
}, '/content/drive/MyDrive/eurosat_model_full.pth')
print("\n✅ Full model checkpoint saved for FastAPI deployment!")

In [ ]:
# Step 1: Install
# !pip install rasterio Pillow numpy --quiet

# import rasterio
# import numpy as np
# from PIL import Image
# import glob
# import os

# # Step 2: Point to your SAFE folder
# # Upload the extracted .SAFE folder to Google Drive first
# SAFE_PATH = "/content/drive/MyDrive/Dataset/S2A_MSIL2A_20180601_YOUR_SCENE.SAFE"

# # Step 3: Auto-find the RGB band files inside .SAFE
# b04_path = glob.glob(f"{SAFE_PATH}/**/B04_10m.jp2", recursive=True)
# b03_path = glob.glob(f"{SAFE_PATH}/**/B03_10m.jp2", recursive=True)
# b02_path = glob.glob(f"{SAFE_PATH}/**/B02_10m.jp2", recursive=True)

# print("Red  (B04):", b04_path)
# print("Green(B03):", b03_path)
# print("Blue (B02):", b02_path)

# # Step 4: Read the bands
# def read_band(path):
#     with rasterio.open(path) as src:
#         return src.read(1).astype(np.float32)

# red   = read_band(b04_path[0])
# green = read_band(b03_path[0])
# blue  = read_band(b02_path[0])

print(f"Band shape: {red.shape}")   # e.g. (10980, 10980)

# Step 5: Normalize to 0-255 (Sentinel-2 values go up to ~10000)
def normalize(band, min_val=0, max_val=3000):
    band = np.clip(band, min_val, max_val)
    return ((band / max_val) * 255).astype(np.uint8)

rgb = np.dstack([normalize(red), normalize(green), normalize(blue)])

# Step 6: Save as proper visible image
img = Image.fromarray(rgb)
img.save("/content/drive/MyDrive/sentinel_rgb_visible.jpg")
print(f"✅ Saved! Size: {img.size}")

# Quick preview in Colab
import matplotlib.pyplot as plt
plt.figure(figsize=(10, 10))
plt.imshow(img)
plt.axis('off')
plt.title("Sentinel-2 RGB — your region")
plt.show()